In [ ]:
from enum import Enum

class MessageType(Enum):
    PERIODIC_REPORT = 1
    PROPOSED_FLIGHT_PLAN = 2
    

class Message:
    def __init__(self, source, destination,message_type,payload,issue_timestamp):
        self.source = source
        self.destination = destination
        self.message_type = message_type
        self.payload = payload
        self.issue_timestamp = issue_timestamp

In [89]:
content = {
    "x": 100,
    "y": 200,
    "z": 300,
}
msg = Message("node1","node2","periodicReport",content,0)
print(msg.payload)

{'x': 100, 'y': 200, 'z': 300}


In [90]:
import simpy

class CommunicationNetwork:
    def __init__(self, env, delay=0,capacity=10000):
        self.env = env
        self.delay = delay
        self.channel = simpy.Resource(env, capacity=capacity)
        self.nodes = {}

    def register(self,node):
        # Enroll communication node into the network.
        # Communication node is a simpy process that can represent a UAV, a UTM module (e.g, deconfliction module, tracking module, ...) etc
        self.nodes[node.name] = node
    
    def send(self,message):
        return self.env.process(self._transmit(message))

    def _transmit(self,message):
        with self.channel.request() as request:
            yield request

            print(
                f"t{self.env.now}: {message.source} sent "
                f"{message.message_type} to {message.destination} "
                f"Payload: {message.payload}"
            )

            yield self.env.timeout(self.delay)

            destination = self.nodes[message.destination]

            # Every node has an inbox attribute to deal with message passing
            yield destination.inbox.put(message)

            print(
                f"t{self.env.now}: {message.message_type} from {message.source} delivered "
                f"to {message.destination} "
                f"Payload: {message.payload}"
            )


In [91]:
import simpy

class GenericNode:
    def __init__(self,name,env, network):
        self.name = name
        self.inbox = simpy.Store(env)
        network.register(self)

    def send_message(self, message):
        network.send(message)


env = simpy.Environment()
network = CommunicationNetwork(env, delay=10)

node1 = GenericNode("node1",env,network)
node2 = GenericNode("node2",env,network)

node1.send_message(msg)

env.run()

t0: node1 sent periodicReport to node2 Payload: {'x': 100, 'y': 200, 'z': 300}
t10: periodicReport from node1 delivered to node2 Payload: {'x': 100, 'y': 200, 'z': 300}


In [ ]:
import simpy

class UAV:
    def __init__(self, name, env, network, position, velocity):
        self.name = name
        self.env = env
        self.network = network
        self.velocity = velocity
        self.inbox = simpy.Store(env)
        self.position = position
    
        network.register(self)

    def update_position(self, time_step):
        while True:
            yield self.env.timeout(time_step)

            self.position[0] += self.velocity[0] * time_step
            self.position[1] += self.velocity[1] * time_step
            self.position[2] += self.velocity[2] * time_step

            # print(
            #     f"t{self.env.now}: {self.name} position updated "
            #     f"to {self.position}"
            # )
    
    def send_periodic_reports(self,tracking_module, update_interval):
        while True:
            msg = Message(
                source=self.name,
                destination=tracking_module,
                message_type="PERIODIC_REPORT",
                payload={
                    "position": self.position.copy(),
                    "velocity": self.velocity.copy()
                },
                issue_timestamp=self.env.now
            )

            yield self.network.send(msg)
            yield self.env.timeout(update_interval)

    def send_flight_plan(self,strategic_deconfliction_module, flight_plan):
        msg = Message(
                source=self.name,
                destination=strategic_deconfliction_module,
                message_type="PROPOSED_FLIGHT_PLAN",
                payload=flight_plan,
                issue_timestamp=self.env.now
            )

    def receive_messages(self):
        pass

In [93]:
class TrackingModule:
    def __init__(self, name, env, network, processing_time):
        self.name = name
        self.env = env
        self.network = network
        self.inbox = simpy.Store(env)
        self.processing_time = processing_time
        network.register(self)

    
    def run(self):
        while True:
            msg = yield self.inbox.get()

            print(
                f"t{self.env.now}: {self.name} processing "
                f"{msg.message_type} from {msg.source}"
            )

            yield self.env.timeout(self.processing_time)

            if msg.message_type == "PERIODIC_REPORT":
                self.process_report(msg)

    def process_report(self, message):
        print(
            f"t{self.env.now}: position of "
            f"{message.source} updated"
        )


In [94]:
env = simpy.Environment()

network = CommunicationNetwork(env=env, delay = 2)

trackingModule = TrackingModule(
    env=env,
    name="TrackingModule",
    network=network,
    processing_time=0.5
)

uav_1 = UAV(
    env=env,
    name="UAV 1",
    network=network,
    position = [0,0,100],
    velocity = [1,1,1]
)

uav_2 = UAV(
    env=env,
    name="UAV 2",
    network=network,
    position = [100,50,100],
    velocity=[4,0,0]
    )

env.process(trackingModule.run())

#env.process(uav_1.receive_messages())
#env.process(uav_2.receive_messages())

env.process(
    uav_1.send_periodic_reports(
        tracking_module="TrackingModule",
        update_interval=2
    )
)

env.process(
    uav_1.update_position(
        time_step=0.1
    )
)

env.process(
    uav_2.send_periodic_reports(
        tracking_module="TrackingModule",
        update_interval=2
    )
)

env.process(
    uav_2.update_position(
        time_step=0.5
    )
)

env.run(until=50)

t0: UAV 1 sent PERIODIC_REPORT to TrackingModule Payload: {'position': [0, 0, 100], 'velocity': [1, 1, 1]}
t0: UAV 2 sent PERIODIC_REPORT to TrackingModule Payload: {'position': [100, 50, 100], 'velocity': [4, 0, 0]}
t2: PERIODIC_REPORT from UAV 1 delivered to TrackingModule Payload: {'position': [0, 0, 100], 'velocity': [1, 1, 1]}
t2: PERIODIC_REPORT from UAV 2 delivered to TrackingModule Payload: {'position': [100, 50, 100], 'velocity': [4, 0, 0]}
t2: TrackingModule processing PERIODIC_REPORT from UAV 1
t2.5: position of UAV 1 updated
t2.5: TrackingModule processing PERIODIC_REPORT from UAV 2
t3.0: position of UAV 2 updated
t4: UAV 1 sent PERIODIC_REPORT to TrackingModule Payload: {'position': [3.900000000000002, 3.900000000000002, 103.89999999999978], 'velocity': [1, 1, 1]}
t4: UAV 2 sent PERIODIC_REPORT to TrackingModule Payload: {'position': [114.0, 50.0, 100.0], 'velocity': [4, 0, 0]}
t6: PERIODIC_REPORT from UAV 1 delivered to TrackingModule Payload: {'position': [3.900000000000

In [ ]:
class TacticalDeconflictionModule:
    def __init__(self, name, env, network):
        self.name = name
        self.env = env
        self.network = network
        self.inbox = simpy.Store(env)
        network.register(self)

    

    